# BEA_V2 — cross-fitted downstream-utility-aware allocation

POST_W12_EXPLORATORY_BEA_V2. This reused development cohort is NOT fresh validation. Run All audits exact historical labels, runs CPU tests, trains CPU cross-fit models, reports held-out diagnostics, calibrates on three queries, freezes identity, and only then starts missing pilot reader calls. **Do not run the full pilot while V1 is using the GPU.** SFR/query fragments, tokenizer, reader, prompt and decoding remain frozen.


In [ ]:
from pathlib import Path
import sys, subprocess, json
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
while not (ROOT/'src/post_w12/bea_v2').is_dir():
    if ROOT == ROOT.parent: raise FileNotFoundError('Open inside the project')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
RUN_PILOT = True
RUN_UNSEEN_EXPANSION = False
RUN_NEW_ATTRIBUTION = False
AUTO_RUN_MISSING = True
REUSE_COMPATIBLE_CACHE = True
RUN_MISSING = True
RESUME = True
OVERWRITE = False
FORCE_REGENERATE = False
if OVERWRITE or FORCE_REGENERATE or not RESUME: raise ValueError('Immutable results require resume and no overwrite')
from src.post_w12.bea_v2.protocol import load, root, freeze
from src.post_w12.bea_v2.data import audit
from src.post_w12.bea_v2.models import train
from src.post_w12.bea_v2.stages import calibrate, prepare, generate
from src.post_w12.bea_v2.baselines import restore_controls
from src.post_w12.bea_v2.analysis import analyze
cfg = load()
if not AUTO_RUN_MISSING or not RUN_MISSING or not REUSE_COMPATIBLE_CACHE:
    raise ValueError('This notebook uses automatic immutable resume; leave reuse/missing-work flags enabled')
OUT = root(cfg)


## 1. Exact historical attribution audit
Only checksum-frozen historical labels and exact source-unit identities are accepted. Passage/clause labels are not transferred to sentences. Any ambiguous mapping stops execution.


In [ ]:
training_units, cohort = audit(cfg)
display(Markdown((OUT/'AUDIT_ATTRIBUTION_DATA.md').read_text()))


## 2. CPU unit tests
These tests use synthetic features, tiny logistic models and source metadata. They do not generate answers or load SFR/Qwen.


In [ ]:
import os
test_env = dict(os.environ, CUDA_VISIBLE_DEVICES='', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
subprocess.run([str(ROOT/'sara_env/bin/python'), '-m', 'pytest', str(ROOT/'W12-POST/bea_v2_tests'), '-q'],cwd=ROOT,env=test_env,check=True)


## 3–5. Frozen five-fold group cross-fitting and held-out predictor diagnostics
Per-dataset StandardScaler + two L2 logistic models, C=1. All outcomes for a held-out query/paper/component remain outside its predictor. No hyperparameter search. Weak predictive performance is reported directly.


In [ ]:
utility_manifest = train(cfg)
display(Markdown((OUT/'utility_model/UTILITY_MODEL_REPORT.md').read_text()))


## 6–7. Three-query allocation-only calibration and identity freeze
One query per dataset; both budgets, fixed threshold grid and mechanical rule. No answers or held-out LOO labels enter allocation calibration. NEEDS_REVIEW blocks generation; do not relax the rule after examining outcomes.


In [ ]:
calibration = calibrate(cfg)
display(Markdown((OUT/'calibration/ALLOCATION_CALIBRATION_REPORT.md').read_text()))
parent, v1, splits, cohort, frozen = freeze(cfg)
print('Frozen V2 identity:', frozen['identity_hash'], 'threshold:', frozen['identity']['threshold'])


## 8–10. Resume missing V2 contexts and answers; reuse historical controls
Only missing BEA_V2 rows normally require reader calls (maximum 300 before prompt reuse). Every result is saved immediately. Missing original controls are restored by canonical historical functions into the NEW V2 namespace, never V1. No old algorithm is manually recreated.


In [ ]:
if RUN_PILOT:
    prepare(cfg)
    generate(cfg)
    print(restore_controls(cfg))
else:
    print('Pilot disabled; no reader answers generated.')


## 11–12. Complete developmental comparisons and decision
All 24 dataset/budget/method cells and 18 paired contrasts are reported. Group-aware uncertainty is conditional on fitted cross-fit predictors, not a bootstrap of model retraining. No pooled task score or automatic expansion.


In [ ]:
if RUN_PILOT:
    result = analyze(cfg)
    for name in ['summary','paired_comparisons','evidence_behavior']: display(result[name])
    display(Markdown((OUT/'pilot/PILOT_DECISION.md').read_text()))
    display(Markdown((OUT/'pilot/PILOT_REPORT.md').read_text()))


## Separate unseen-VAL planning — disabled
This opt-in only freezes a metadata cohort plan, excluding all development/training groups, confirmation and any V1-expanded groups. It does not launch unseen inference. All-development predictors and new exact problems require a separately reviewed extension before execution. New attribution is not part of this experiment.


In [ ]:
if RUN_UNSEEN_EXPANSION:
    from src.post_w12.bea_v2.unseen import plan as plan_unseen
    print(json.dumps(plan_unseen(cfg),indent=2))
if RUN_NEW_ATTRIBUTION:
    raise ValueError('New attribution is outside this frozen V2 pilot; use a separately reviewed version')
